# 15 — EDITO JupyterLab post-processing setup

**Run this notebook inside JupyterLab launched from the EDITO Datalab service catalog.** It validates S3 access to your personal bucket and demonstrates how to open the v02 outputs *without downloading* them — the `map.nc` alone is ~16 GB, so streaming via `s3fs` + lazy xarray chunks is the only sustainable workflow.

**Launch procedure** (EDITO UI):
1. Datalab → Service Catalog → search `jupyter` → pick `JupyterLab Python` (or the data-science variant)
2. Start service — S3 credentials are auto-injected as env vars (no manual config)
3. Open a terminal inside JupyterLab and pull the code (see cell 2)
4. Open this notebook and run all cells

## 1. Install any missing packages

Most EDITO JupyterLab images have the scientific stack but not `dfm_tools`/`xugrid`. Install only what's missing.

In [ ]:
import importlib, subprocess, sys
need = []
for pkg in ['xarray', 'xugrid', 's3fs', 'netCDF4', 'matplotlib', 'pandas', 'dfm_tools']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        need.append(pkg)
if need:
    print('Installing:', need)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *need])
else:
    print('All packages present.')

## 2. Pull project code from S3

The code was uploaded by `scripts/edito_sync.py sync-code` (from your laptop). We pull it into the JupyterLab home so the notebooks 08, 09, 12, 13 are available.

Run **once per JupyterLab session** (the home persists but re-syncs ensure you have the latest code).

In [ ]:
import os, boto3, pathlib

BUCKET = 'oidc-cmartinsjr'
CODE_PREFIX = 'CODE/'
HOME = pathlib.Path.home() / 'stagnone'
HOME.mkdir(exist_ok=True)

# On EDITO services, these env vars are auto-set (endpoint may lack https://)
endpoint = os.environ.get('AWS_S3_ENDPOINT', 'minio.dive.edito.eu')
if not endpoint.startswith('http'):
    endpoint = 'https://' + endpoint

s3 = boto3.client('s3',
    endpoint_url=endpoint,
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
)

paginator = s3.get_paginator('list_objects_v2')
n = 0
for page in paginator.paginate(Bucket=BUCKET, Prefix=CODE_PREFIX):
    for obj in page.get('Contents', []):
        rel = obj['Key'][len(CODE_PREFIX):]
        if not rel:
            continue
        local = HOME / rel
        local.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, obj['Key'], str(local))
        n += 1
print(f'Pulled {n} files into {HOME}')

## 3. Sanity check — list your bucket

In [ ]:
def list_prefix(prefix, limit=10):
    paginator = s3.get_paginator('list_objects_v2')
    out = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get('Contents', [])[:limit]:
            out.append((obj['Key'], obj['Size']))
        if len(out) >= limit:
            break
    return out

for pfx in ['DFM_INPUT/', 'DFM_OUTPUT/', 'CODE/']:
    items = list_prefix(pfx)
    print(f'\n{pfx}  ({len(items)} shown):')
    for k, sz in items:
        print(f'  {sz/1e6:>10.2f} MB   {k}')

## 4. Open `map.nc` from S3 WITHOUT downloading

Two proven patterns:

- **A (preferred): `s3fs` + `xarray.open_dataset`** — S3-backed file object, chunked reads via h5netcdf. Works when the file is regular NetCDF4/HDF5.
- **B (fallback): stream-to-tempfile then open** — if chunked reads on HDF5 over S3 are too slow/erratic.

For D-Flow FM `map.nc` (NetCDF4/HDF5), approach A is the right one.

In [ ]:
import s3fs, xarray as xr

# Reuse `endpoint` normalized in cell-4 (cells run top-to-bottom)
fs = s3fs.S3FileSystem(
    key=os.environ['AWS_ACCESS_KEY_ID'],
    secret=os.environ['AWS_SECRET_ACCESS_KEY'],
    token=os.environ.get('AWS_SESSION_TOKEN'),
    client_kwargs={'endpoint_url': endpoint},
)

# List outputs
output_keys = [p for p in fs.ls(f'{BUCKET}/DFM_OUTPUT/') if p.endswith(('_map.nc', '_his.nc'))]
print('Found:', output_keys)

In [ ]:
import xugrid as xu

his_key = next(k for k in output_keys if k.endswith('_his.nc'))
map_keys = sorted(k for k in output_keys if k.endswith('_map.nc'))
print(f'his:  {his_key}')
print(f'map partitions ({len(map_keys)}):')
for k in map_keys:
    print(f'  {k}')

# NetCDF3 classic (ncFormat=3) → use scipy engine (not h5netcdf which requires HDF5)
xr_parts = [xr.open_dataset(fs.open(k, 'rb'), engine='scipy', chunks={'time': 50})
            for k in map_keys]
ug_parts = [xu.UgridDataset(ds) for ds in xr_parts]
ds_map = xu.UgridDataset.merge_partitions(ug_parts)
print(f'\nmap merged  dims: {dict(ds_map.sizes)}')
print(f'            vars: {list(ds_map.data_vars)[:8]} ...')
print(f'            time: {ds_map.time.values[0]} .. {ds_map.time.values[-1]}')

In [ ]:
# his.nc is small (~30 MB) — safe to load eagerly
with fs.open(his_key, mode='rb') as f:
    ds_his = xr.open_dataset(f.read())   # reads once into memory via bytes
# Workaround for engines that don't accept bytes: use BytesIO
import io
with fs.open(his_key, mode='rb') as f:
    ds_his = xr.open_dataset(io.BytesIO(f.read()), engine='h5netcdf')
print(f'his.nc  dims: {dict(ds_his.sizes)}')
print(f'        stations: {[s.decode().strip() if isinstance(s, bytes) else str(s).strip() for s in ds_his.station_name.values]}')

## 5. Quick sanity plot — time-averaged water depth

Triggers one pass over the full time dimension; with `chunks={'time':50}` this stays under ~500 MB RAM even for the full 16 GB map.nc.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

wd_mean = ds_map['mesh2d_waterdepth'].mean(dim='time').compute()
grid = ds_map.ugrid.grids[0]
fx, fy = grid.face_coordinates[:, 0], grid.face_coordinates[:, 1]

fig, ax = plt.subplots(figsize=(7, 9))
sc = ax.scatter(fx, fy, c=wd_mean.values, s=1, cmap='viridis')
plt.colorbar(sc, label='Mean water depth (m)')
ax.set_aspect(1/np.cos(np.radians(37.87)))
ax.set_title('Time-averaged water depth — v02')
plt.tight_layout(); plt.show()

## 6. Next steps on EDITO

- Run existing post-processing notebooks **08** and **09** — paths need to point to S3 objects via `fs.open(...)` instead of local Path. Easiest: add 2 cells at top of those notebooks to set `ds_map = xr.open_dataset(fs.open(map_key), ...)` and `ds_his = ...`, then use those datasets downstream.
- Outputs of analysis (CSV, PNG) → save to JupyterLab home (`/home/onyxia/work/...`) and download at the end, **or** push back to S3 with `boto3.upload_file` under a `RESULTS/` prefix.
- For v03 residence-time runs (≥30 days) the `map.nc` can reach 50-90 GB — **never download it**, only use streaming as done here.
- Close `fobj` and `ds_map.close()` at end of notebook to release the S3 connection.